In [3]:
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    ViTForImageClassification,
    ViTImageProcessor,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import (
    Compose, Resize, CenterCrop, RandomHorizontalFlip,
    RandomRotation, ColorJitter, ToTensor, Normalize
)
from torch.optim import Adam
from torch.optim.lr_scheduler import MultiplicativeLR
import evaluate

# ── 1. Load & combine all data ────────────────────────────────────────────────
ds_raw   = load_dataset("hf-vision/chest-xray-pneumonia")
full_data = concatenate_datasets([ds_raw["train"], ds_raw["validation"], ds_raw["test"]])

print(f"Total images: {len(full_data)}")
labels = full_data["label"]
print(f"PNEUMONIA (1): {labels.count(1)}")
print(f"NORMAL    (0): {labels.count(0)}")

# ── 2. Recreate paper's 80/10/10 stratified split ────────────────────────────
split1      = full_data.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_ds_raw = split1["train"]

split2      = split1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
val_ds_raw  = split2["train"]
test_ds_raw = split2["test"]

print(f"\nOur split:")
print(f"Train: {len(train_ds_raw)} — P:{train_ds_raw['label'].count(1)} N:{train_ds_raw['label'].count(0)}")
print(f"Val:   {len(val_ds_raw)}  — P:{val_ds_raw['label'].count(1)} N:{val_ds_raw['label'].count(0)}")
print(f"Test:  {len(test_ds_raw)} — P:{test_ds_raw['label'].count(1)} N:{test_ds_raw['label'].count(0)}")

print(f"\nPaper split:")
print(f"Train: 4684 — P:3205 N:1479")
print(f"Val:   586  — P:360  N:226")
print(f"Test:  586  — P:330  N:256")

# ── 3. Processor & transforms ─────────────────────────────────────────────────
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")

MEAN = processor.image_mean
STD  = processor.image_std
SIZE = processor.size["height"]  # 224

train_transforms = Compose([
    Resize((SIZE, SIZE)),
    RandomHorizontalFlip(),
    RandomRotation(10),
    ColorJitter(brightness=0.2, contrast=0.2),
    ToTensor(),
    Normalize(mean=MEAN, std=STD),
])

eval_transforms = Compose([
    Resize((SIZE, SIZE)),
    CenterCrop(SIZE),
    ToTensor(),
    Normalize(mean=MEAN, std=STD),
])

def apply_train_transforms(batch):
    batch["pixel_values"] = [
        train_transforms(img.convert("RGB")) for img in batch["image"]
    ]
    return batch

def apply_eval_transforms(batch):
    batch["pixel_values"] = [
        eval_transforms(img.convert("RGB")) for img in batch["image"]
    ]
    return batch

train_ds = train_ds_raw.with_transform(apply_train_transforms)
val_ds   = val_ds_raw.with_transform(apply_eval_transforms)
test_ds  = test_ds_raw.with_transform(apply_eval_transforms)

# ── 4. Model ──────────────────────────────────────────────────────────────────
id2label = {0: "NORMAL", 1: "PNEUMONIA"}
label2id = {"NORMAL": 0, "PNEUMONIA": 1}

model = ViTForImageClassification.from_pretrained(
    "google/vit-base-patch16-224",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# ── 5. Metrics ────────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]

    tp = np.sum((preds == 1) & (labels == 1))
    tn = np.sum((preds == 0) & (labels == 0))
    fp = np.sum((preds == 1) & (labels == 0))
    fn = np.sum((preds == 0) & (labels == 1))

    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)

    return {
        "accuracy":    round(acc, 4),
        "sensitivity": round(float(sensitivity), 4),
        "specificity": round(float(specificity), 4),
    }

# ── 6. Collator ───────────────────────────────────────────────────────────────
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels":       torch.tensor([x["label"] for x in batch]),
    }

# ── 7. Training arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./vit-pneumonia-papersplit",
    num_train_epochs=15,
    per_device_train_batch_size=16,        # paper: 16
    per_device_eval_batch_size=16,
    learning_rate=1e-5,                    # paper: 1e-5
    weight_decay=0.0,                      # paper: plain Adam, no decay
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
)

# ── 8. Custom Trainer — Adam + MultiplicativeLR matching paper ────────────────
class PaperTrainer(Trainer):
    def create_optimizer_and_scheduler(self, num_training_steps):
        self.optimizer = Adam(self.model.parameters(), lr=1e-5)
        self.lr_scheduler = MultiplicativeLR(
            self.optimizer,
            lr_lambda=lambda epoch: 0.995  # paper: multiplicative factor 0.995
        )

trainer = PaperTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# ── 9. Train ──────────────────────────────────────────────────────────────────
trainer.train()

# ── 10. Final evaluation on test set ─────────────────────────────────────────
print("\n── Test Set Results ──")
results = trainer.evaluate(test_ds)
print(results)

# ── 11. Save ──────────────────────────────────────────────────────────────────
trainer.save_model("./vit-pneumonia-papersplit")
processor.save_pretrained("./vit-pneumonia-papersplit")
print("\nModel saved to ./vit-pneumonia-papersplit")

Total images: 5856
PNEUMONIA (1): 4273
NORMAL    (0): 1583

Our split:
Train: 4684 — P:3418 N:1266
Val:   586  — P:427 N:159
Test:  586 — P:428 N:158

Paper split:
Train: 4684 — P:3205 N:1479
Val:   586  — P:360  N:226
Test:  586  — P:330  N:256


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Epoch,Training Loss,Validation Loss,Accuracy,Sensitivity,Specificity
1,0.086807,0.141162,0.953900,0.967200,0.918200
2,0.104831,0.170533,0.948800,0.955500,0.930800
3,0.067207,0.153585,0.953900,0.960200,0.937100
4,0.057847,0.137334,0.959000,0.962500,0.949700
5,0.066121,0.142498,0.960800,0.953200,0.981100
6,0.060130,0.126204,0.964200,0.981300,0.918200
7,0.079728,0.108242,0.965900,0.976600,0.937100
8,0.039184,0.123686,0.972700,0.971900,0.974800
9,0.034426,0.122066,0.971000,0.971900,0.968600
10,0.014215,0.139550,0.965900,0.978900,0.930800


...


── Test Set Results ──


Training Loss,Validation Loss,Epoch,Accuracy,Sensitivity,Specificity
0.005305,0.156335,15,0.959000,0.957900,0.962000


{'eval_loss': 0.156334787607193, 'eval_accuracy': 0.959, 'eval_sensitivity': 0.9579, 'eval_specificity': 0.962}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./vit-pneumonia-papersplit


In [ ]:
import numpy as np
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer
)
from torchvision.transforms import (
    Compose, Resize, CenterCrop, RandomHorizontalFlip,
    RandomRotation, ColorJitter, ToTensor, Normalize
)
from torch.optim import Adam
from torch.optim.lr_scheduler import MultiplicativeLR
import evaluate

# ── 1. Load & combine all data ────────────────────────────────────────────────
ds_raw    = load_dataset("hf-vision/chest-xray-pneumonia")
full_data = concatenate_datasets([ds_raw["train"], ds_raw["validation"], ds_raw["test"]])

print(f"Total images: {len(full_data)}")
labels = full_data["label"]
print(f"PNEUMONIA (1): {labels.count(1)}")
print(f"NORMAL    (0): {labels.count(0)}")

# ── 2. Recreate paper's 80/10/10 stratified split ────────────────────────────
split1       = full_data.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
train_ds_raw = split1["train"]

split2      = split1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
val_ds_raw  = split2["train"]
test_ds_raw = split2["test"]

print(f"\nOur split:")
print(f"Train: {len(train_ds_raw)} — P:{train_ds_raw['label'].count(1)} N:{train_ds_raw['label'].count(0)}")
print(f"Val:   {len(val_ds_raw)}  — P:{val_ds_raw['label'].count(1)} N:{val_ds_raw['label'].count(0)}")
print(f"Test:  {len(test_ds_raw)} — P:{test_ds_raw['label'].count(1)} N:{test_ds_raw['label'].count(0)}")

print(f"\nPaper split:")
print(f"Train: 4684 — P:3205 N:1479")
print(f"Val:   586  — P:360  N:226")
print(f"Test:  586  — P:330  N:256")

# ── 3. Processor & transforms ─────────────────────────────────────────────────
processor = AutoImageProcessor.from_pretrained("facebook/deit-base-patch16-224")

MEAN = processor.image_mean
STD  = processor.image_std
SIZE = processor.size["height"]  # 224

train_transforms = Compose([
    Resize((SIZE, SIZE)),
    RandomHorizontalFlip(),
    RandomRotation(10),
    ColorJitter(brightness=0.2, contrast=0.2),
    ToTensor(),
    Normalize(mean=MEAN, std=STD),
])

eval_transforms = Compose([
    Resize((SIZE, SIZE)),
    CenterCrop(SIZE),
    ToTensor(),
    Normalize(mean=MEAN, std=STD),
])

def apply_train_transforms(batch):
    batch["pixel_values"] = [
        train_transforms(img.convert("RGB")) for img in batch["image"]
    ]
    return batch

def apply_eval_transforms(batch):
    batch["pixel_values"] = [
        eval_transforms(img.convert("RGB")) for img in batch["image"]
    ]
    return batch

train_ds = train_ds_raw.with_transform(apply_train_transforms)
val_ds   = val_ds_raw.with_transform(apply_eval_transforms)
test_ds  = test_ds_raw.with_transform(apply_eval_transforms)

# ── 4. Model ──────────────────────────────────────────────────────────────────
id2label = {0: "NORMAL", 1: "PNEUMONIA"}
label2id = {"NORMAL": 0, "PNEUMONIA": 1}

model = AutoModelForImageClassification.from_pretrained(
    "facebook/deit-base-patch16-224",
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

# ── 5. Metrics ────────────────────────────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]

    tp = np.sum((preds == 1) & (labels == 1))
    tn = np.sum((preds == 0) & (labels == 0))
    fp = np.sum((preds == 1) & (labels == 0))
    fn = np.sum((preds == 0) & (labels == 1))

    sensitivity = tp / (tp + fn + 1e-8)
    specificity = tn / (tn + fp + 1e-8)

    return {
        "accuracy":    round(acc, 4),
        "sensitivity": round(float(sensitivity), 4),
        "specificity": round(float(specificity), 4),
    }

# ── 6. Collator ───────────────────────────────────────────────────────────────
def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels":       torch.tensor([x["label"] for x in batch]),
    }

# ── 7. Training arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./deit-pneumonia-papersplit",
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-5,
    weight_decay=0.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=True,
    dataloader_num_workers=0,
    remove_unused_columns=False,
    report_to="none",
)

# ── 8. Custom Trainer — Adam + MultiplicativeLR matching paper ────────────────
class PaperTrainer(Trainer):
    def create_optimizer_and_scheduler(self, num_training_steps):
        self.optimizer = Adam(self.model.parameters(), lr=1e-5)
        self.lr_scheduler = MultiplicativeLR(
            self.optimizer,
            lr_lambda=lambda epoch: 0.995
        )

trainer = PaperTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# ── 9. Train ──────────────────────────────────────────────────────────────────
trainer.train()

# ── 10. Final evaluation on test set ─────────────────────────────────────────
print("\n── Test Set Results ──")
results = trainer.evaluate(test_ds)
print(results)

# ── 11. Save ──────────────────────────────────────────────────────────────────
trainer.save_model("./deit-pneumonia-papersplit")
processor.save_pretrained("./deit-pneumonia-papersplit")
print("\nModel saved to ./deit-pneumonia-papersplit")

Total images: 5856
PNEUMONIA (1): 4273
NORMAL    (0): 1583

Our split:
Train: 4684 — P:3418 N:1266
Val:   586  — P:427 N:159
Test:  586 — P:428 N:158

Paper split:
Train: 4684 — P:3205 N:1479
Val:   586  — P:360  N:226
Test:  586  — P:330  N:256


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


pytorch_model.bin:   0%|          | 0.00/346M [00:00<?, ?B/s]

...


── Test Set Results ──


Training Loss,Validation Loss,Epoch,Accuracy,Sensitivity,Specificity
0.002845,0.154320,15,0.965900,0.969600,0.955700


{'eval_loss': 0.15431973338127136, 'eval_accuracy': 0.9659, 'eval_sensitivity': 0.9696, 'eval_specificity': 0.9557}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to ./deit-pneumonia-papersplit


In [1]:
import os
from datasets import load_dataset, concatenate_datasets

# Recreate the same split (same seed = same images)
ds_raw    = load_dataset("hf-vision/chest-xray-pneumonia")
full_data = concatenate_datasets([ds_raw["train"], ds_raw["validation"], ds_raw["test"]])

split1      = full_data.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
split2      = split1["test"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
test_ds_raw = split2["test"]

# Save some samples to disk
os.makedirs("sample_xrays/NORMAL", exist_ok=True)
os.makedirs("sample_xrays/PNEUMONIA", exist_ok=True)

normal_saved    = 0
pneumonia_saved = 0

for i, sample in enumerate(test_ds_raw):
    label = sample["label"]
    image = sample["image"]

    if label == 0 and normal_saved < 10:
        image.save(f"sample_xrays/NORMAL/normal_{normal_saved+1}.jpeg")
        normal_saved += 1
    elif label == 1 and pneumonia_saved < 10:
        image.save(f"sample_xrays/PNEUMONIA/pneumonia_{pneumonia_saved+1}.jpeg")
        pneumonia_saved += 1

    if normal_saved == 10 and pneumonia_saved == 10:
        break

print(f"Saved {normal_saved} NORMAL and {pneumonia_saved} PNEUMONIA images")
print("Location: sample_xrays/")

Saved 10 NORMAL and 10 PNEUMONIA images
Location: sample_xrays/
